In [ ]:
from pyspark.sql import SparkSession

try:
    spark.stop()
except:
    pass

spark = SparkSession.builder \
    .appName("Build_Hive_Tables_Final") \
    .config("spark.driver.memory", "6g") \
    .enableHiveSupport() \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

spark.sql("CREATE DATABASE IF NOT EXISTS ecommerce_db")
spark.sql("USE ecommerce_db")

base = "/home/lst/my-spark/star_schema/"   # ←←← 改成你实际路径（最后加 / ）

print("开始创建 Hive 表...")

# 1. 事实表（专业版，带注释）
spark.sql(f"""
CREATE EXTERNAL TABLE IF NOT EXISTS fact_order_items (
    item_id           STRING COMMENT '订单明细ID',
    increment_id      STRING COMMENT '订单号',
    created_at        TIMESTAMP COMMENT '下单时间',
    price             DECIMAL(10,2) COMMENT '单价',
    qty_ordered       INT COMMENT '购买数量',
    total_value       DECIMAL(12,2) COMMENT '行总金额',
    grand_total       DECIMAL(12,2) COMMENT '订单总金额',
    discount_amount   DECIMAL(10,2) COMMENT '折扣金额',
    status            STRING COMMENT '订单状态',
    payment_method    STRING COMMENT '支付方式',
    sku               STRING COMMENT '商品SKU',
    `Customer ID`     STRING COMMENT '客户ID',
    category_name_1   STRING COMMENT '一级品类'
)
COMMENT '电商订单事实表 - 订单明细粒度（星型模型核心表）'
PARTITIONED BY (year INT, month INT)
STORED AS PARQUET
LOCATION '{base}fact_order_items'
""")

# 2. 维度表
spark.sql(f"CREATE EXTERNAL TABLE IF NOT EXISTS dim_products STORED AS PARQUET LOCATION '{base}dim_products' AS SELECT * FROM parquet.`{base}dim_products`")
spark.sql(f"CREATE EXTERNAL TABLE IF NOT EXISTS dim_customers STORED AS PARQUET LOCATION '{base}dim_customers' AS SELECT * FROM parquet.`{base}dim_customers`")
spark.sql(f"CREATE EXTERNAL TABLE IF NOT EXISTS dim_time STORED AS PARQUET LOCATION '{base}dim_time' AS SELECT * FROM parquet.`{base}dim_time`")
spark.sql(f"CREATE EXTERNAL TABLE IF NOT EXISTS dim_status STORED AS PARQUET LOCATION '{base}dim_status' AS SELECT * FROM parquet.`{base}dim_status`")

print("✅ 所有 Hive 表已创建完成")


In [2]:
from pyspark.sql import SparkSession

try:
    spark.stop()
except:
    pass

spark = SparkSession.builder \
    .appName("Rebuild_All_Tables_Final") \
    .enableHiveSupport() \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

spark.sql("USE ecommerce_db")

base = "/home/lst/my-spark/star_schema/"   # ←←← 改成你实际路径，最后加 /

print("开始从零重建并加载所有表...")

# 删除旧表（避免残留问题）
for tbl in ["fact_order_items", "dim_products", "dim_customers", "dim_time", "dim_status"]:
    spark.sql(f"DROP TABLE IF EXISTS {tbl}")

print("旧表已清理")

# 1. 创建并加载维度表（最简单方式）
for tbl in ["dim_products", "dim_customers", "dim_time", "dim_status"]:
    spark.sql(f"""
        CREATE TABLE {tbl} 
        STORED AS PARQUET 
        AS SELECT * FROM parquet.`{base}{tbl}`
    """)
    print(f"✅ {tbl} 创建并加载完成")

# 2. 创建并加载事实表（分区表）
spark.sql(f"""
    CREATE TABLE fact_order_items 
    PARTITIONED BY (year, month)
    STORED AS PARQUET 
    AS SELECT 
        item_id,
        increment_id,
        created_at,
        price,
        qty_ordered,
        total_value,
        grand_total,
        discount_amount,
        status,
        payment_method,
        sku,
        `Customer ID`,
        category_name_1,
        year,
        month
    FROM parquet.`{base}fact_order_items`
""")

print("✅ fact_order_items 创建并加载完成")

# 刷新分区
spark.sql("MSCK REPAIR TABLE fact_order_items")

print("\n🎉 所有表重建并加载完成！")

# 验证
spark.sql("""
SELECT 'fact_order_items' as table_name, count(*) as row_count FROM fact_order_items
UNION ALL
SELECT 'dim_products', count(*) FROM dim_products
UNION ALL
SELECT 'dim_customers', count(*) FROM dim_customers
UNION ALL
SELECT 'dim_time', count(*) FROM dim_time
UNION ALL
SELECT 'dim_status', count(*) FROM dim_status
""").show()

开始从零重建并加载所有表...
旧表已清理
✅ dim_products 创建并加载完成
✅ dim_customers 创建并加载完成
✅ dim_time 创建并加载完成
✅ dim_status 创建并加载完成


Py4JJavaError: An error occurred while calling o92.sql.
: org.apache.spark.SparkException: Dynamic partition strict mode requires at least one static partition column. To turn this off set hive.exec.dynamic.partition.mode=nonstrict
	at org.apache.spark.sql.hive.execution.V1WritesHiveUtils.getDynamicPartitionColumns(V1WritesHiveUtils.scala:82)
	at org.apache.spark.sql.hive.execution.V1WritesHiveUtils.getDynamicPartitionColumns$(V1WritesHiveUtils.scala:45)
	at org.apache.spark.sql.hive.execution.InsertIntoHiveTable$.getDynamicPartitionColumns(InsertIntoHiveTable.scala:288)
	at org.apache.spark.sql.hive.execution.InsertIntoHiveTable$.apply(InsertIntoHiveTable.scala:316)
	at org.apache.spark.sql.hive.execution.CreateHiveTableAsSelectCommand.getWritingCommand(CreateHiveTableAsSelectCommand.scala:107)
	at org.apache.spark.sql.hive.execution.CreateHiveTableAsSelectCommand.run(CreateHiveTableAsSelectCommand.scala:82)
	at org.apache.spark.sql.execution.command.ExecutedCommandExec.sideEffectResu

In [3]:
spark.sql("USE ecommerce_db")

print("=== 当前 4 个维度表 + 事实表行数验证 ===")
spark.sql("""
SELECT 'dim_products'     as table_name, count(*) as row_count FROM dim_products
UNION ALL
SELECT 'dim_customers'    as table_name, count(*) FROM dim_customers
UNION ALL
SELECT 'dim_time'         as table_name, count(*) FROM dim_time
UNION ALL
SELECT 'dim_status'       as table_name, count(*) FROM dim_status
UNION ALL
SELECT 'fact_order_items' as table_name, count(*) FROM fact_order_items
""").show()

=== 当前 4 个维度表 + 事实表行数验证 ===


AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `fact_order_items` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.; line 10 pos 55;
'Union false, false
:- Union false, false
:  :- Union false, false
:  :  :- Union false, false
:  :  :  :- Aggregate [dim_products AS table_name#129, count(1) AS row_count#130L]
:  :  :  :  +- SubqueryAlias spark_catalog.ecommerce_db.dim_products
:  :  :  :     +- Relation spark_catalog.ecommerce_db.dim_products[sku#140,category_name_1#141] parquet
:  :  :  +- Aggregate [dim_customers AS table_name#131, count(1) AS count(1)#148L]
:  :  :     +- SubqueryAlias spark_catalog.ecommerce_db.dim_customers
:  :  :        +- Relation spark_catalog.ecommerce_db.dim_customers[Customer ID#142,Customer Since